# Phase 1: Data Cleaning & Pre-processing
This notebook processes the raw dataset, handling missing values via KNN Imputation *before* mapping strings/categories.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer

In [ ]:
df = pd.read_csv('data/Dataset_maternal_mental_health_infant_sleep.csv', encoding='unicode_escape')
df.head()

,Participant_number,Type_parents,Birth_1mth_M_inclusion,Birth_12mth_M_inclusion,Age,Marital_status,Marital_status_edit,Education,Gestationnal_age,Type_pregnancy,...,IBQ_R_VSF_10_bb1,IBQ_R_VSF_16_bb1,IBQ_R_VSF_17_bb1,IBQ_R_VSF_28_bb1,IBQ_R_VSF_29_bb1,IBQ_R_VSF_32_bb1,IBQ_R_VSF_33_bb1,Sleep_night_duration_bb1,night_awakening_number_bb1,how_falling_asleep_bb1
0,1,1,1,1,34,2,2,5,37.0,1,...,NaN,7.0,NaN,NaN,7.0,6.0,NaN,10:00,3,2
1,2,1,1,1,33,2,2,5,42.0,1,...,2.0,3.0,2.0,2.0,2.0,3.0,4.0,11:00,0,4
2,3,1,1,1,37,2,2,5,41.0,1,...,4.0,4.0,3.0,1.0,4.0,NaN,NaN,12:00,1,2
3,4,1,1,1,31,2,2,5,37.5,1,...,1.0,3.0,NaN,NaN,NaN,5.0,NaN,11:00,2,1
4,5,1,1,1,36,1,1,5,40.0,1,...,4.0,2.0,2.0,4.0,5.0,6.0,6.0,10:30,1,4


In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

### 1. Drop Irrelevant Columns
Removing columns that do not add predictive value or duplicate information.

In [4]:
df = df.set_index('Participant_number')

df = df.drop(columns=['Type_parents', 'Birth_1mth_M_inclusion', 'Birth_12mth_M_inclusion',
                 'Marital_status', 'Child_survey_participation'])
df.columns = df.columns.str.lower()

### 2. Parse Sleep Duration
The `sleep_night_duration_bb1` column is currently a string (hh:mm). The value `99:99` represents a missing value.
We must convert these strings into decimal hours (e.g., 10:30 -> 10.5) so the KNN imputer can process them.

In [5]:
def parse_sleep_duration(time_str):
    if pd.isna(time_str) or time_str == '99:99':
        return np.nan
    try:
        parts = str(time_str).split(':')
        if len(parts) == 2:
            return int(parts[0]) + (int(parts[1]) / 60.0)
        return np.nan
    except:
        return np.nan

In [6]:

# Apply the parsing function
df['sleep_night_duration_bb1'] = df['sleep_night_duration_bb1'].apply(parse_sleep_duration)
print(f"Missing sleep values to impute: {df['sleep_night_duration_bb1'].isna().sum()}")

Missing sleep values to impute: 1


### 3. KNN Imputation
Because survey answers are numerical integers under the hood, we must run the KNN imputer *before* altering column types to strings/categories.

In [7]:
def column_knn_imputer(df):
    imputer = KNNImputer(n_neighbors=5)
    # Select only numeric columns that have missing values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cols_with_missing = [c for c in numeric_cols if df[c].isnull().any()]
    
    if not cols_with_missing:
        return df.copy()

    df_imputed = df.copy()
    df_imputed[cols_with_missing] = imputer.fit_transform(df[cols_with_missing])
    
    # Round everything except numerical/time back to whole integers for survey answers
    cols_to_round = [c for c in cols_with_missing if c not in ['gestationnal_age', 'sleep_night_duration_bb1']]
    if cols_to_round:
        df_imputed[cols_to_round] = df_imputed[cols_to_round].round()
    
    return df_imputed

In [8]:

df_clean = column_knn_imputer(df)
print(f"Remaining missing numeric values: {df_clean.select_dtypes(include=[np.number]).isna().sum().sum()}")

Remaining missing numeric values: 0


### 4. Text Mapping & Categorical Conversions
Now that imputation is complete mathematically, we can map numerical flags to text and cast survey components to `category` dtypes for EDA plotting.

In [9]:
# Text Mapping
df_clean['marital_status_edit'] = df_clean['marital_status_edit'].replace({1: 'single', 2: 'in a relationship', 3: 'separated-divorced-widow'})
df_clean['type_pregnancy'] = df_clean['type_pregnancy'].replace({1: 'single', 2: 'twin'})
df_clean['age_bb'] = df_clean['age_bb'].replace({1: '3 to <6', 2: '6 to <9', 3: '9 to <12'})
df_clean['how_falling_asleep_bb1'] = df_clean['how_falling_asleep_bb1'].replace({
    1: 'while being fed', 2: 'while being rocked',
    3: 'while being held', 4: 'alone in the crib',
    5: 'in the crib with parental presence'
})
df_clean['education'] = df_clean['education'].replace({1: 'no education', 2: 'high school', 3: 'some university', 4: 'associate/tech', 5: 'university'})
df_clean['sex_baby1'] = df_clean['sex_baby1'].replace({1: 'girl', 2: 'boy'})

# Clean up remaining column names
df_clean = df_clean.rename(columns={'marital_status_edit': 'marital_status', 'sex_baby1': 'sex'})

In [10]:
# Cast demographics to categorical dtype
cat_cols = ['marital_status', 'education', 'type_pregnancy', 'sex', 'age_bb', 'how_falling_asleep_bb1']
for col in cat_cols:
    df_clean[col] = df_clean[col].astype('category')
    

In [11]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 410 entries, 1 to 410
Data columns (total 57 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   age                         410 non-null    int64   
 1   marital_status              410 non-null    category
 2   education                   410 non-null    category
 3   gestationnal_age            410 non-null    float64 
 4   type_pregnancy              410 non-null    category
 5   sex                         410 non-null    category
 6   cbts_m_3                    410 non-null    int64   
 7   cbts_m_4                    410 non-null    int64   
 8   cbts_m_5                    410 non-null    int64   
 9   cbts_m_6                    410 non-null    int64   
 10  cbts_m_7                    410 non-null    int64   
 11  cbts_m_8                    410 non-null    int64   
 12  cbts_m_9                    410 non-null    int64   
 13  cbts_m_10                   410

In [12]:
df_clean.head()

,age,marital_status,education,gestationnal_age,type_pregnancy,sex,cbts_m_3,cbts_m_4,cbts_m_5,cbts_m_6,cbts_m_7,cbts_m_8,cbts_m_9,cbts_m_10,cbts_m_11,cbts_m_12,cbts_13,cbts_14,cbts_15,cbts_16,cbts_17,cbts_18,cbts_19,cbts_20,cbts_21,cbts_22,epds_1,epds_2,epds_3,epds_4,epds_5,epds_6,epds_7,epds_8,epds_9,epds_10,hads_1,hads_3,hads_5,hads_7,hads_9,hads_11,hads_13,age_bb,ibq_r_vsf_3_bb1,ibq_r_vsf_4_bb1,ibq_r_vsf_9_bb1,ibq_r_vsf_10_bb1,ibq_r_vsf_16_bb1,ibq_r_vsf_17_bb1,ibq_r_vsf_28_bb1,ibq_r_vsf_29_bb1,ibq_r_vsf_32_bb1,ibq_r_vsf_33_bb1,sleep_night_duration_bb1,night_awakening_number_bb1,how_falling_asleep_bb1
Participant_number,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,34,in a relationship,university,37.0,single,girl,0,0,0,0,0,0,0,1,0,0,0,0,0,0,2,0,2,0,0,1,1,2,2,1,1,2,0,2,2,0,2,1,2,2,0,1,1,3 to <6,6.0,3.0,7.0,6.0,7.0,4.0,3.0,7.0,6.0,4.0,10.0,3,while being rocked
2,33,in a relationship,university,42.0,single,boy,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,9 to <12,1.0,4.0,2.0,2.0,3.0,2.0,2.0,2.0,3.0,4.0,11.0,0,alone in the crib
3,37,in a relationship,university,41.0,single,girl,0,0,1,0,0,0,0,0,0,1,1,0,0,0,2,0,2,0,2,0,1,0,2,1,0,2,0,1,1,0,2,1,1,2,0,3,0,3 to <6,1.0,3.0,5.0,4.0,4.0,3.0,1.0,4.0,4.0,2.0,12.0,1,while being rocked
4,31,in a relationship,university,37.5,single,boy,0,0,1,1,1,0,0,1,1,1,2,0,1,0,1,0,2,0,2,0,1,1,2,2,2,2,2,2,1,1,1,3,3,1,2,2,1,9 to <12,2.0,5.0,2.0,1.0,3.0,2.0,3.0,3.0,5.0,5.0,11.0,2,while being fed
5,36,single,university,40.0,single,boy,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1,0,0,2,0,0,0,0,1,0,0,1,0,1,0,9 to <12,2.0,6.0,2.0,4.0,2.0,2.0,4.0,5.0,6.0,6.0,10.5,1,alone in the crib


In [13]:
# Export the cleaned dataset for Phase 1 (EDA) & Phase 4 (Scoring)
df_clean.to_csv('Dataset_Cleaned_Imputed.csv', index=False)
print('Data successfully exported to Dataset_Cleaned_Imputed.csv')

Data successfully exported to Dataset_Cleaned_Imputed.csv
